# Lab6.3 — Procesare video și evaluarea performanței end-to-end

**Aplicație Edge:** achiziționăm un cadru video, aplicăm Resize → Grayscale → Sobel și calculăm un indicator al densității muchiilor.

Comparăm două trasee care pot fi executate cu resursele existente:

- **A / ARM:** toate operațiile OpenCV sunt efectuate pe ARM.
- **B / hibrid:** Resize pe FPGA cu overlay-ul din Lab5, apoi Grayscale și Sobel pe ARM.

**Pipeline-ul complet FPGA Resize → Sobel** a fost analizat în Lab6.2, dar nu dispune încă de un overlay verificat. Nu vom prezenta simulări sau estimări drept măsurători FPGA.

**Rezultate urmărite:** timpul achiziției, timpul procesării, latența totală și percentila 95, FPS-ul efectiv al buclei, analiza blocajelor și un experiment cu parametri modificați.

## 1. Alegerea sursei și protocolul experimental

La început utilizați `MODE = "synthetic"`, care funcționează fără cameră. Ulterior puteți selecta:

- `"file"` — fișier video local;
- `"camera"` — cameră USB, de exemplu Logitech C270.

**Precizare:** sursa sintetică generează cadre prin software, deci timpul măsurat aici nu este timp real de captură de la o cameră.

Pentru fiecare traseu deschidem din nou sursa. Cadrele sintetice sunt reproductibile, dar două experimente succesive cu o cameră USB nu vor prelua exact aceleași imagini.

In [ ]:
from pathlib import Path
from time import perf_counter
import cv2
import numpy as np
import matplotlib.pyplot as plt
from lab6_common import (load_test_image, locate_resizer, ResizerHW,
                         process_cpu, process_hybrid)

MODE = "synthetic"        # "synthetic", "file" sau "camera"
VIDEO_FILE = Path("video.mp4")  # utilizat doar pentru MODE="file"
CAMERA_INDEX = 0
N_FRAMES = 50
TARGET = (320, 240)
EDGE_THRESHOLD = 80  # pragul de intensitate pentru muchii

class VideoSource:
    def __init__(self, mode):
        self.mode = mode
        self.index = 0
        self.cap = None
        if mode == "synthetic":
            self.base = cv2.resize(load_test_image(), (640, 480),
                                   interpolation=cv2.INTER_LINEAR)
        elif mode in ("file", "camera"):
            if mode == "file" and not VIDEO_FILE.is_file():
                raise FileNotFoundError("Lipseste fisierul video: " + str(VIDEO_FILE))
            self.cap = cv2.VideoCapture(str(VIDEO_FILE) if mode == "file"
                                        else CAMERA_INDEX)
            if not self.cap.isOpened():
                raise RuntimeError("Nu se poate deschide sursa %s." % mode)
            if mode == "camera":
                self.cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
                self.cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
        else:
            raise ValueError("MODE trebuie sa fie synthetic, file sau camera")

    def read(self):
        if self.mode == "synthetic":
            h, w = self.base.shape[:2]
            dx = (self.index * 7) % 51 - 25
            dy = (self.index * 3) % 31 - 15
            M = np.float32([[1, 0, dx], [0, 1, dy]])
            frame = cv2.warpAffine(self.base, M, (w, h),
                                   borderMode=cv2.BORDER_REFLECT_101)
            self.index += 1
            return True, frame
        ok, frame = self.cap.read()
        if not ok and self.mode == "file":
            self.cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
            ok, frame = self.cap.read()
        return ok, frame

    def close(self):
        if self.cap is not None:
            self.cap.release()

print("Sursa:", MODE, "Cadre:", N_FRAMES, "Rezolutie iesire:", TARGET)

## 2. Măsurarea buclei complete de procesare

Separăm următoarele componente:

1. **Sursă:** capturarea sau generarea unui cadru.
2. **Procesare:** Resize → Grayscale → Sobel.
3. **Decizie:** determinarea fracției pixelilor cu gradient ridicat.
4. **End-to-end:** toate operațiile de la începutul achiziției până la decizie.
5. **FPS efectiv:** numărul cadrelor procesate împărțit la durata reală a buclei.

Nu afișăm fiecare cadru în interiorul secvenței cronometrate, deoarece randarea în browser ar introduce costuri suplimentare.

In [ ]:
def run_application(path, processor, n_frames=N_FRAMES):
    source = VideoSource(MODE)
    samples = []
    preview = None
    t_start = perf_counter()
    try:
        for _ in range(n_frames):
            t0 = perf_counter()
            ok, frame = source.read()
            t1 = perf_counter()
            if not ok:
                break
            edges = processor(frame)
            t2 = perf_counter()
            edge_fraction = float(np.mean(edges >= EDGE_THRESHOLD))
            t3 = perf_counter()
            if preview is None:
                preview = (frame.copy(), edges.copy())
            samples.append((1000*(t1-t0), 1000*(t2-t1),
                            1000*(t3-t2), 1000*(t3-t0), edge_fraction))
    finally:
        source.close()
    total_seconds = perf_counter() - t_start
    if not samples:
        raise RuntimeError("Nu s-au obtinut cadre valide din sursa " + MODE)
    data = np.asarray(samples)
    report = dict(path=path, count=len(samples),
                  source_mean_ms=float(np.mean(data[:, 0])),
                  process_mean_ms=float(np.mean(data[:, 1])),
                  decision_mean_ms=float(np.mean(data[:, 2])),
                  e2e_mean_ms=float(np.mean(data[:, 3])),
                  e2e_p95_ms=float(np.percentile(data[:, 3], 95)),
                  observed_loop_fps=float(len(samples)/total_seconds),
                  edge_fraction_mean=float(np.mean(data[:, 4])))
    return report, preview

arm_report, arm_preview = run_application(
    "A / ARM", lambda frame: process_cpu(frame, TARGET))
print("Varianta A:", arm_report)

## 3. Execuția hibridă pe PYNQ-Z2

În această etapă folosim **același overlay Resize** ca în Lab5. Configurarea inițială și programarea FPGA sunt excluse din bucla repetată, dar copierea datelor, DMA, Resize, Grayscale și Sobel sunt incluse.

Executați mai întâi Lab6.1 și verificați că traseul hibrid funcționează stabil la 320×240. În caz de blocare DMA, opriți și reporniți kernelul; nu repetați execuția fără depanare.

Dacă overlay-ul lipsește, afișăm explicit faptul că traseul B nu a fost măsurat.

In [ ]:
hw = None
hybrid_report = None
hybrid_preview = None
try:
    bit = locate_resizer()
    hw = ResizerHW(bit)
    print("Fisierul bit al variantei hibride:", bit)
    hybrid_report, hybrid_preview = run_application(
        "B / hybrid", lambda frame: process_hybrid(frame, TARGET, hw))
    print("Varianta B:", hybrid_report)
except (ImportError, FileNotFoundError, AttributeError, RuntimeError) as exc:
    print("Varianta B nemasurata (hardware sau sursa indisponibila):", exc)
finally:
    if hw is not None:
        hw.close()
        hw = None

reports = [arm_report] + ([hybrid_report] if hybrid_report else [])
print("\n%-14s %11s %11s %11s %11s %11s" %
      ("Traseu", "Sursa ms", "Proc. ms", "Decizie ms", "E2E ms", "FPS bucla"))
for r in reports:
    print("%-14s %11.2f %11.2f %11.2f %11.2f %11.2f" %
          (r["path"], r["source_mean_ms"], r["process_mean_ms"],
           r["decision_mean_ms"], r["e2e_mean_ms"], r["observed_loop_fps"]))

## 4. Vizualizarea rezultatelor

Un accelerator FPGA cu throughput ridicat nu garantează un FPS ridicat al întregii aplicații. Sursa video, transferurile DMA, procesarea ARM și decizia finală pot domina timpul total.

Într-o buclă sincronă, FPS-ul observat poate fi influențat și de frecvența cu care sursa livrează cadre. Raportăm separat latența medie per cadru și FPS-ul observat în timpul experimentului.

In [ ]:
preview = hybrid_preview if hybrid_preview is not None else arm_preview
src, edges = preview
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].imshow(cv2.cvtColor(src, cv2.COLOR_BGR2RGB))
axes[0].set_title("Cadru achizitionat/generat")
axes[1].imshow(edges, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Muchii Resize + Sobel")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

x = np.arange(len(reports))
width = 0.22
plt.figure(figsize=(8, 4))
for offset, key, name in [(-width, "source_mean_ms", "Source"),
                          (0, "process_mean_ms", "Processing"),
                          (width, "decision_mean_ms", "Decision")]:
    plt.bar(x+offset, [r[key] for r in reports], width=width, label=name)
plt.xticks(x, [r["path"] for r in reports])
plt.ylabel("Durata medie masurata [ms/cadru]")
plt.title("Componentele latentei / sursa=%s" % MODE)
plt.legend(); plt.tight_layout(); plt.show()

## 5. Experiment integrator și concluzii

Repetați măsurătorile pentru **două rezoluții de ieșire**. Pentru început efectuați această variație pe ARM; încercați o altă rezoluție pe FPGA numai după validarea DMA. Dacă aveți o cameră USB, comparați sursa sintetică și sursa reală.

În raport răspundeți:

1. Ce proporție din timpul end-to-end reprezintă achiziția și ce proporție procesarea?
2. Se obține un FPS mai mare cu varianta hibridă? Justificați cu valori măsurate.
3. De ce nu se poate raporta FPS-ul sursei sintetice drept performanță a camerei USB?
4. Ce costuri ar reduce un pipeline FPGA Resize → Sobel complet și ce costuri ar rămâne?
5. Propuneți un scenariu Edge Computing în care reducerea traficului DDR are importanță practică.

**Raportați statutul experimental:** A și B sunt măsurate unde hardware-ul permite; configurația C (pipeline complet PL) este deocamdată un proiect analizat în Lab6.2.